In [1]:
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import torch.optim as optim

import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

In [2]:
# nltk.download("punkt")
# nltk.download("punkt_tab")
# nltk.download("stopwords")

# Load and inspect

In [3]:
rs_df = pd.read_csv(
    "datasets/review_sentiment.csv"
)
df = rs_df.copy()

In [4]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   review     50000 non-null  str  
 1   sentiment  50000 non-null  str  
dtypes: str(2)
memory usage: 781.4 KB


# Preprocessing

### Missing and duplicate values

No need to care for inconsistent values.

In [6]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [7]:
df.duplicated().sum()

np.int64(418)

Duplicates should be removed

In [8]:
df = df.drop_duplicates()

### Cleaning

In [9]:
# To lower case
df["review"] = df["review"].str.lower()

In [10]:
# Removing URLs
def remove_url(text):
    return re.sub(r"http\S+", "", text)

df["review"] = df["review"].map(remove_url)

In [11]:
# Removing punctuation
def remove_puntucation(text):
    return re.sub(r"[^A-Za-z0-9\s]", "", text)

df["review"] = df["review"].map(remove_puntucation)

In [12]:
# Remove html
# Removing punctuation
def remove_html(text):
    return re.sub(r"<.*?>", "", text)

df["review"] = df["review"].map(remove_html)

### Tokenize

In [13]:
df["review"] = df["review"].map(word_tokenize)

### Remove stopwords

In [14]:
stop_words = stopwords.words("english")
def remove_stopwords(tokens):
    filtered = []
    for token in tokens:
        if token not in stop_words:
            filtered.append(token)

    return filtered

df["review"] = df["review"].map(remove_stopwords)
        

### Stemming

In [15]:
stemmer = PorterStemmer()
def stem(tokens):
    stemmed = []
    for token in tokens:
        stemmed.append(stemmer.stem(token))

    return stemmed

df["review"] = df["review"].map(stem)

### Vectorization

In [16]:
# vectorizer expects string docs so we need a dummy function
def dummy(doc):
    return doc

vectorizer = TfidfVectorizer(
    max_features=5000,
    tokenizer=dummy,
    preprocessor=dummy,
    lowercase=False,
    token_pattern=None
)

X = vectorizer.fit_transform(df["review"])

### Encoding output

In [17]:
le = LabelEncoder()
y = le.fit_transform(df["sentiment"])

In [18]:
le.classes_

array(['negative', 'positive'], dtype=object)

### Convert to tensor -> dataset -> dataloader

In [19]:
# Split to train, val and test dataset
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=42
)

In [20]:
# convert to numpy array
X_train = X_train.toarray()
X_val = X_val.toarray()
X_test = X_test.toarray()

In [21]:
# to tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32)

In [22]:
# to dataset
train_set = TensorDataset(X_train_tensor, y_train_tensor)
val_set = TensorDataset(X_val_tensor, y_val_tensor)
test_set = TensorDataset(X_test_tensor, y_test_tensor)

In [23]:
# to dataloaders
train_loader = DataLoader(train_set, shuffle=True, batch_size=64)
val_loader = DataLoader(val_set, shuffle=True, batch_size=64)
test_loader = DataLoader(test_set, shuffle=True, batch_size=64)

# RNN Model

## Define Model

In [1]:
class RNN(nn.Module):

    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.rnn = nn.RNN(
            input_size, # number of features in input
            hidden_size, # number of neurons in each hidden layer
            num_layers, # number of hidden layers
            batch_first=True
        )

        self.fc = nn.Linear(
            hidden_size,
            1
        )

    def forward(self, x):
        # It will be initialized only during the first time
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

        out, _ = self.rnn(x, h0) # out => hidden state of time stamps (batch_size, seq_len, hidden_size)

        return self.fc(out[:, -1, :])




NameError: name 'nn' is not defined

## Training

In [25]:
rnn_model = RNN(
    X_train.shape[1]
)

criterion = nn.BCEWithLogitsLoss()

optimizer = optim.Adam(rnn_model.parameters())

In [26]:
epochs = 10
min_val_loss = float("inf")

for epoch in range(0, epochs):
    rnn_model.train()
    train_loss = 0

    for Xb, yb in train_loader:
        optimizer.zero_grad()
        Xb = Xb.unsqueeze(1) # add extra dimension to make it 3d
        outputs = rnn_model(Xb) # predict logits
        y_pred = torch.sigmoid(outputs.squeeze()) # convert to probabilities 
        loss = criterion(y_pred, yb) # loss calculation
        loss.backward() # back propagation
        optimizer.step() # gradient descent

        train_loss += loss.item()

    rnn_model.eval()
    val_loss = 0

    with torch.no_grad():
        for Xb, yb in val_loader:
            Xb = Xb.unsqueeze(1) # add extra dimension to make it 3d
            outputs = rnn_model(Xb) # predict logits
            y_pred = torch.sigmoid(outputs.squeeze()) # convert to probabilities 
            loss = criterion(y_pred, yb) # loss calculation

            val_loss += loss.item()

    train_loss /= len(train_loader)
    val_loss /= len(val_loader)

    if val_loss < min_val_loss:
        min_val_loss = val_loss
        torch.save( rnn_model.state_dict(), "rnn_sentiment_analysis.pt",)

    print(f"Epoch {epoch} => Train Loss: {train_loss} Val Loss: {val_loss}")

    

Epoch 0 => Train Loss: 0.6039140249692625 Val Loss: 0.5666188347724176
Epoch 1 => Train Loss: 0.5514277738909568 Val Loss: 0.5613831829640173
Epoch 2 => Train Loss: 0.5421773805132797 Val Loss: 0.5625186802879456
Epoch 3 => Train Loss: 0.5367738886345779 Val Loss: 0.5622576564069717
Epoch 4 => Train Loss: 0.5328712208016265 Val Loss: 0.5648751999101331
Epoch 5 => Train Loss: 0.5298070249057585 Val Loss: 0.5649606388422751
Epoch 6 => Train Loss: 0.52753313570734 Val Loss: 0.5651033261129933
Epoch 7 => Train Loss: 0.5258700532177764 Val Loss: 0.5654011277421829
Epoch 8 => Train Loss: 0.5246421082486068 Val Loss: 0.5650867983218162
Epoch 9 => Train Loss: 0.5236259017500186 Val Loss: 0.5669842730126073


## Evaluation

In [27]:
rnn_model.load_state_dict(torch.load("rnn_sentiment_analysis.pt"))

C:\Users\User\AppData\Local\Temp\ipykernel_16112\573956182.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  rnn_model.load_state_dict(torch.load("rnn_sentiment_analysis.p

<All keys matched successfully>

In [28]:
rnn_model.eval()

y_actual = []
y_pred = []

with torch.no_grad():

    for Xb, yb in test_loader:
        Xb = Xb.unsqueeze(1)

        y_pred_logits = rnn_model(Xb)
        y_pred_probs = torch.sigmoid(y_pred_logits.squeeze())
        y_pred_labels = (y_pred_probs >= 0.5).float()

        y_actual.extend(yb.cpu().numpy())
        y_pred.extend(y_pred_labels.cpu().numpy())

In [29]:
print(f"Accuracy: {accuracy_score(y_actual, y_pred)}")
print(classification_report(y_actual, y_pred))

Accuracy: 0.8833316527175558
              precision    recall  f1-score   support

         0.0       0.86      0.91      0.89      4939
         1.0       0.90      0.86      0.88      4978

    accuracy                           0.88      9917
   macro avg       0.88      0.88      0.88      9917
weighted avg       0.88      0.88      0.88      9917



# Console program 

In [88]:
def preprocess(x):
    cleaned = x.lower()
    cleaned = remove_url(cleaned)
    cleaned = remove_puntucation(cleaned)
    cleaned = remove_html(cleaned)
    tokens = word_tokenize(cleaned)
    tokens = remove_stopwords(tokens)
    stemmed_tokens = stem(tokens)
    vector = vectorizer.transform([stemmed_tokens])
    array = vector.toarray()
    return torch.tensor(array, dtype=torch.float32)


In [91]:
rrn_model = RNN(1)
rnn_model.load_state_dict(torch.load("rnn_sentiment_analysis.pt"))

def predict_sentiment(x):
    x = preprocess(x)
    x = x.unsqueeze(0)
    label = rnn_model(x)
    if label[0] <= 0:
        return "Negative"
    else:
        return "Positive"

C:\Users\User\AppData\Local\Temp\ipykernel_16112\146976948.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  rnn_model.load_state_dict(torch.load("rnn_sentiment_analysis.p

In [92]:
while True:
    x = input("\n \n Enter text or 'q' if want to stop the program: ")
    if x == "q":
        break
    else:
        print(f"Sentiment: {predict_sentiment(x)}")

Sentiment: Negative
Sentiment: Positive
Sentiment: Positive
Sentiment: Positive
Sentiment: Negative
Sentiment: Negative
Sentiment: Positive
Sentiment: Positive
Sentiment: Negative
Sentiment: Negative
Sentiment: Negative
